In [2]:
import pandas as pd
import numpy as np
import os

---

### **| Fase 1 ) -** Pré-Processamento

---

In [3]:
df_fair = pd.read_excel('Data/05_Sheets.xlsx')
df_fair.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-001,Statlog (German Credit Data),Link,UCI Repository,Ofc,144,NaN
1,CRED-002,South German Credit,Link,Kaggle,Mod,sid321axn/south-german-credit-updated,CRED-001
2,CRED-003,Default of Credit Card Clients,Link,UCI Repository,Ofc,350,NaN
3,CRED-004,Australian Credit Approval,Link,UCI Repository,No,143,NaN
4,CRED-005,Japanese Credit Screening,Link,UCI Repository,No,28,NaN


In [4]:
df_cleaned = df_fair.dropna(subset=['url']).copy()
df_cleaned = df_cleaned[df_cleaned['is_duplicate'] != 'Yes']
df_cleaned['parent_id'] = df_cleaned['parent_id'].fillna('NaN')
df_cleaned = df_cleaned.reset_index(drop=True)

print(f"| Processamento concluído")
print(f"| Linhas originais: {len(df_fair)}")
print(f"| Linhas prontas para download: {len(df_cleaned)}")

| Processamento concluído
| Linhas originais: 70
| Linhas prontas para download: 60


---

##### | 1.1 ) - Kaggle

In [5]:
df_cleaned_kaggle = df_cleaned[df_cleaned['repo'] == 'Kaggle']
df_cleaned_kaggle.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
1,CRED-002,South German Credit,Link,Kaggle,Mod,sid321axn/south-german-credit-updated,CRED-001
17,CRED-022,Home Credit Default Risk,Link,Kaggle,No,competitions/home-credit-default-risk/data,NaN
18,CRED-023,Lending Club Loan Data,Link,Kaggle,Ofc,wordsforthewise/lending-club,NaN
19,CRED-024,Give Me Some Credit,Link,Kaggle,No,competitions/GiveMeSomeCredit/data,NaN
20,CRED-025,IEEE-CIS Fraud Detection,Link,Kaggle,No,competitions/ieee-fraud-detection/data,NaN


---

##### | 1.2 ) - UCI Repository	

In [6]:
df_cleaned_UCI = df_cleaned[df_cleaned['repo'] == 'UCI Repository']
df_cleaned_UCI.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-001,Statlog (German Credit Data),Link,UCI Repository,Ofc,144,NaN
2,CRED-003,Default of Credit Card Clients,Link,UCI Repository,Ofc,350,NaN
3,CRED-004,Australian Credit Approval,Link,UCI Repository,No,143,NaN
4,CRED-005,Japanese Credit Screening,Link,UCI Repository,No,28,NaN
5,CRED-006,Credit Approval Dataset,Link,UCI Repository,Ofc,27,NaN


---

##### | 1.3 ) - Open ML

In [7]:
df_cleaned_OpenML = df_cleaned[df_cleaned['repo'] == 'OpenML']
df_cleaned_OpenML.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
7,CRED-008,Qualitative Bankruptcy,Link,OpenML,No,1495,NaN
8,CRED-012,default-of-credit-card-clients v1,Link,OpenML,Mod,42477,CRED-003
9,CRED-013,default-of-credit-card-clients v2,Link,OpenML,Mod,45016,CRED-003
10,CRED-014,default-of-credit-card-clients v3,Link,OpenML,Mod,45020,CRED-003
11,CRED-015,default-of-credit-card-clients v4,Link,OpenML,Mod,45036,CRED-003


---

### **| Fase 2 ) -** Implementando API

---

In [8]:
from kaggle.api.kaggle_api_extended import KaggleApi
import kaggle
import shutil

In [9]:
kaggle.api.authenticate()

In [10]:
KAGGLE_BASE_URL = "https://www.kaggle.com/datasets/"

In [11]:
def get_columns_size_mean(list_colums: list) -> float:
    return np.mean([len(str(col)) for col in list_colums])

def is_encrypted(x: float, threshold: int = 5) -> bool:
    return x < threshold

---

##### | 1.1 ) - Implementando para Kaggle

In [12]:
import pandas as pd
import numpy as np
import os
import shutil
import zipfile
import locale
from kaggle.api.kaggle_api_extended import KaggleApi

# CORREÇÃO 1: Força o padrão de tempo para inglês para que a API do Kaggle não quebre no pt_BR
try:
    locale.setlocale(locale.LC_TIME, 'C')
except locale.Error:
    pass # Ignora caso o sistema já esteja no formato correto

def process_kaggle_datasets(df_input, path='./DataBuffer'):
    api = KaggleApi()
    api.authenticate()

    if not os.path.exists(path):
        os.makedirs(path)

    results_list = []
    processed_cache = {}

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        nome = row['nome']
        url = row['url']
        is_dup = row['is_duplicate']
        
        parent_id = str(row['parent_id']).replace('[', '').replace(']', '')

        if is_dup == 'Yes' and parent_id in processed_cache:
            print(f"  -> Duplicata detectada. Copiando metadados da matriz {parent_id}.")
            cached_data = processed_cache[parent_id].copy()
            cached_data['id'] = dataset_id
            cached_data['Dataset_Title'] = nome
            cached_data['User_Ref'] = ref
            cached_data['URL'] = url
            cached_data['is_duplicate'] = is_dup
            cached_data['parent_id'] = parent_id
            
            results_list.append(cached_data)
            continue 

        try:
            # CORREÇÃO 2: Lógica separada para descompactar competições
            if ref.startswith('c/'):
                comp_name = ref.replace('c/', '')
                # Baixa sem o argumento unzip (que causava o erro)
                api.competition_download_files(comp_name, path=path)
                
                # Descompacta manualmente arquivos .zip encontrados no buffer
                for item in os.listdir(path):
                    if item.endswith('.zip'):
                        zip_ref = zipfile.ZipFile(os.path.join(path, item), 'r')
                        zip_ref.extractall(path)
                        zip_ref.close()
                        os.remove(os.path.join(path, item)) # Deleta o zip limpo
            else:
                # Datasets normais aceitam o unzip nativo
                api.dataset_download_files(ref, path=path, unzip=True)
                
            files = [f for f in os.listdir(path) if f.endswith('.csv')]
            
            if not files:
                print("\n|  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *\n")
                print(f"|  -> Nenhum arquivo CSV encontrado para '{ref}'.")
                print("\n|  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *  *\n")
                continue
                
            file_path = os.path.join(path, files[0])
            df_temp = pd.read_csv(file_path, low_memory=False)
            
            sensitive_terms = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status']
            found_sensitive = [col for col in df_temp.columns if any(s in str(col).lower() for s in sensitive_terms)]
            
            columns = df_temp.columns.values

            row_data = {
                'id': dataset_id,
                'User_Ref': ref,
                'Dataset_Title': nome,
                'URL': url,
                'is_duplicate': is_dup,
                'parent_id': parent_id if parent_id != 'nan' else 'NaN',
                'File_Name': files[0],
                'is_encrypted' : is_encrypted(x=get_columns_size_mean(columns), threshold=5),
                'Columns' : columns,
                'Columns_Count': df_temp.shape[1],
                'Rows_Count': df_temp.shape[0],
                'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            }
            
            results_list.append(row_data)
            
            if is_dup in ['Ofc', 'No']:
                processed_cache[dataset_id] = row_data
                
        except Exception as e:
            print("\n|")
            print(f"|  -> Erro ao processar o link Kaggle '{ref}': {e}")
            print("|\n")
        
        finally:
            for f in os.listdir(path):
                file_to_rem = os.path.join(path, f)
                try:
                    if os.path.isfile(file_to_rem):
                        os.remove(file_to_rem)
                    elif os.path.isdir(file_to_rem):
                        shutil.rmtree(file_to_rem)
                except Exception:
                    pass

    return pd.DataFrame(results_list)

df_info_kaggle = process_kaggle_datasets(df_input=df_cleaned_kaggle)

Dataset URL: https://www.kaggle.com/datasets/sid321axn/south-german-credit-updated
Dataset URL: https://www.kaggle.com/datasets/competitions/home-credit-default-risk/versions/data

|
|  -> Erro ao processar o link Kaggle 'competitions/home-credit-default-risk/data': invalid literal for int() with base 10: 'data'
|

Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club

|
|  -> Erro ao processar o link Kaggle 'wordsforthewise/lending-club': [Errno 21] Is a directory: './DataBuffer/accepted_2007_to_2018q4.csv'
|

Dataset URL: https://www.kaggle.com/datasets/competitions/GiveMeSomeCredit/versions/data

|
|  -> Erro ao processar o link Kaggle 'competitions/GiveMeSomeCredit/data': invalid literal for int() with base 10: 'data'
|

Dataset URL: https://www.kaggle.com/datasets/competitions/ieee-fraud-detection/versions/data

|
|  -> Erro ao processar o link Kaggle 'competitions/ieee-fraud-detection/data': invalid literal for int() with base 10: 'data'
|

Dataset URL: https:

In [13]:
df_info_kaggle.head()

,id,User_Ref,Dataset_Title,URL,is_duplicate,parent_id,File_Name,is_encrypted,Columns,Columns_Count,Rows_Count,Missing_Values,Numeric_Cols,Categorical_Cols,Sensitive_Columns,Memory_Usage_MB
0,CRED-002,sid321axn/south-german-credit-updated,South German Credit,Link,Mod,CRED-001,german_credit.csv,False,"[status, duration, credit_history, purpose, am...",21,1000,0,3,18,"[status, personal_status_sex, age]",0.40
1,CRED-026,mlg-ulb/creditcardfraud,Credit Card Fraud Detection (ULB),Link,Ofc,NaN,creditcard.csv,True,"[Time, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10...",31,284807,0,31,0,[],67.36
2,CRED-027,laotse/credit-risk-dataset,Credit Risk Dataset (laotse),Link,Ofc,NaN,credit_risk_dataset.csv,False,"[person_age, person_income, person_home_owners...",12,32581,4011,8,4,"[person_age, loan_status]",3.53
3,CRED-028,saurabhbagchi/dish-network-hackathon,Automobile Loan Default Dataset,Link,No,NaN,Test_Dataset.csv,False,"[ID, Client_Income, Car_Owned, Bike_Owned, Act...",39,80900,263377,18,21,"[Client_Marital_Status, Client_Gender, Age_Day...",33.18
4,CRED-029,nelgiriyewithana/credit-card-fraud-detection-d...,Credit Card Fraud Detection 2023,Link,No,NaN,creditcard_2023.csv,True,"[id, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10, ...",31,568630,0,31,0,[],134.49


---

##### | 1.2 ) - Implementando para UCI

In [14]:
from ucimlrepo import fetch_ucirepo

def process_uci_datasets(df_input):
    results_list = []
    processed_cache = {}

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        nome = row['nome']
        url = row['url']
        is_dup = row['is_duplicate']
        
        parent_id = str(row['parent_id']).replace('[', '').replace(']', '')

        if is_dup == 'Yes' and parent_id in processed_cache:
            print(f"  -> Duplicata detectada. Copiando metadados da matriz {parent_id}.")
            cached_data = processed_cache[parent_id].copy()
            cached_data['id'] = dataset_id
            cached_data['Dataset_Title'] = nome
            cached_data['User_Ref'] = ref
            cached_data['URL'] = url
            cached_data['is_duplicate'] = is_dup
            cached_data['parent_id'] = parent_id
            
            results_list.append(cached_data)
            continue 

        try:
            # Baixa o dataset via API oficial da UCI usando o ID
            dataset = fetch_ucirepo(id=int(ref))
            
            # Pega o DataFrame completo (Features + Targets)
            df_temp = dataset.data.original
            
            # Proteção caso a API da UCI falhe em entregar o dataframe unificado
            if df_temp is None:
                df_temp = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
            
            sensitive_terms = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status']
            found_sensitive = [col for col in df_temp.columns if any(s in str(col).lower() for s in sensitive_terms)]
            
            columns = df_temp.columns.values

            row_data = {
                'id': dataset_id,
                'User_Ref': ref,
                'Dataset_Title': nome,
                'URL': url,
                'is_duplicate': is_dup,
                'parent_id': parent_id if parent_id != 'nan' else 'NaN',
                'File_Name': 'UCI_API_Direct', # Não há arquivo físico
                'is_encrypted' : is_encrypted(x=get_columns_size_mean(columns), threshold=5),
                'Columns' : columns,
                'Columns_Count': df_temp.shape[1],
                'Rows_Count': df_temp.shape[0],
                'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            }
            
            results_list.append(row_data)
            
            if is_dup in ['Ofc', 'No']:
                processed_cache[dataset_id] = row_data
                
        except Exception as e:
            print("\n|")
            print(f"|  -> Erro ao processar o link UCI '{ref}': {e}")
            print("|\n")

    return pd.DataFrame(results_list)

# Execução:
df_info_uci = process_uci_datasets(df_input=df_cleaned_UCI)


|
|  -> Erro ao processar o link UCI '144': Error connecting to server
|


|
|  -> Erro ao processar o link UCI '350': Error connecting to server
|


|
|  -> Erro ao processar o link UCI '143': Error connecting to server
|


|
|  -> Erro ao processar o link UCI '28': Error connecting to server
|


|
|  -> Erro ao processar o link UCI '27': Error connecting to server
|


|
|  -> Erro ao processar o link UCI '365': Error connecting to server
|



In [15]:
df_info_uci

""


---

##### | 1.3 ) - Implementando para Open ML

In [16]:
import openml

def process_openml_datasets(df_input):
    results_list = []
    processed_cache = {}

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        nome = row['nome']
        url = row['url']
        is_dup = row['is_duplicate']
        
        parent_id = str(row['parent_id']).replace('[', '').replace(']', '')

        if is_dup == 'Yes' and parent_id in processed_cache:
            print(f"  -> Duplicata detectada. Copiando metadados da matriz {parent_id}.")
            cached_data = processed_cache[parent_id].copy()
            cached_data['id'] = dataset_id
            cached_data['Dataset_Title'] = nome
            cached_data['User_Ref'] = ref
            cached_data['URL'] = url
            cached_data['is_duplicate'] = is_dup
            cached_data['parent_id'] = parent_id
            
            results_list.append(cached_data)
            continue 

        try:
            # Baixa o dataset diretamente para a memória via API do OpenML
            dataset = openml.datasets.get_dataset(int(ref), download_data=True)
            
            # Extrai os dados já em formato Pandas DataFrame
            # X = features, y = target (juntamos os dois para análise completa)
            X, y, _, _ = dataset.get_data(dataset_format='dataframe')
            df_temp = pd.concat([X, y], axis=1)
            
            sensitive_terms = ['age', 'gender', 'sex', 'race', 'ethnicity', 'status']
            found_sensitive = [col for col in df_temp.columns if any(s in str(col).lower() for s in sensitive_terms)]
            
            columns = df_temp.columns.values

            row_data = {
                'id': dataset_id,
                'User_Ref': ref,
                'Dataset_Title': nome,
                'URL': url,
                'is_duplicate': is_dup,
                'parent_id': parent_id if parent_id != 'nan' else 'NaN',
                'File_Name': 'OpenML_API_Direct', # Não há arquivo físico
                'is_encrypted' : is_encrypted(x=get_columns_size_mean(columns), threshold=5),
                'Columns' : columns,
                'Columns_Count': df_temp.shape[1],
                'Rows_Count': df_temp.shape[0],
                'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            }
            
            results_list.append(row_data)
            
            if is_dup in ['Ofc', 'No']:
                processed_cache[dataset_id] = row_data
                
        except Exception as e:
            print("\n|")
            print(f"|  -> Erro ao processar o link OpenML '{ref}': {e}")
            print("|\n")

    return pd.DataFrame(results_list)

# Execução:
df_info_openml = process_openml_datasets(df_input=df_cleaned_OpenML)

In [17]:
df_info_openml.head()

,id,User_Ref,Dataset_Title,URL,is_duplicate,parent_id,File_Name,is_encrypted,Columns,Columns_Count,Rows_Count,Missing_Values,Numeric_Cols,Categorical_Cols,Sensitive_Columns,Memory_Usage_MB
0,CRED-008,1495,Qualitative Bankruptcy,Link,No,NaN,OpenML_API_Direct,True,"[V1, V2, V3, V4, V5, V6, Class]",7,250,0,0,7,[],0.00
1,CRED-012,42477,default-of-credit-card-clients v1,Link,Mod,CRED-003,OpenML_API_Direct,True,"[x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11,...",24,30000,0,23,1,[],4.49
2,CRED-013,45016,default-of-credit-card-clients v2,Link,Mod,CRED-003,OpenML_API_Direct,True,"[x1, x5, x6, x7, x8, x9, x10, x11, x12, x13, x...",21,13272,0,21,0,[],1.95
3,CRED-014,45020,default-of-credit-card-clients v3,Link,Mod,CRED-003,OpenML_API_Direct,True,"[x1, x5, x6, x7, x8, x9, x10, x11, x12, x13, x...",21,13272,0,20,1,[],1.95
4,CRED-015,45036,default-of-credit-card-clients v4,Link,Mod,CRED-003,OpenML_API_Direct,True,"[x1, x2, x5, x6, x7, x8, x9, x10, x11, x12, x1...",22,13272,0,20,2,[],1.96


---

### **| Fase 3 ) -** Finalizando Dataset

---

In [19]:
print("\nConsolidando as bases de dados...")
# df_final = pd.concat([df_info_kaggle, df_info_openml, df_info_uci], ignore_index=True)
# 2. Juntar todos os DataFrames funcionais em um único
print("Consolidando as bases de dados extraídas...")
df_final = pd.concat([df_info_kaggle, df_info_openml], ignore_index=True)

# =====================================================================
# 2. LIMPEZA E FORMATAÇÃO VISUAL
# =====================================================================

# A. Remover as colunas redundantes que não agregam ao artigo
df_final = df_final.drop(columns=['File_Name', 'User_Ref'], errors='ignore')

# B. Limpar a coluna 'Columns' (Forçar o formato visual de lista ['col1', 'col2'])
df_final['Columns'] = df_final['Columns'].apply(
    lambda x: str(list(x)) if hasattr(x, '__iter__') and not isinstance(x, str) else str(x)
)

# =====================================================================
# 3. ENGENHARIA DE ATRIBUTOS (MÉTRICAS PARA O ARTIGO)
# =====================================================================

# C. Taxa de Esparsidade (% de células vazias no dataset inteiro)
df_final['Sparsity_Ratio_%'] = np.where(
    (df_final['Rows_Count'] * df_final['Columns_Count']) > 0,
    round((df_final['Missing_Values'] / (df_final['Rows_Count'] * df_final['Columns_Count'])) * 100, 4),
    0
)

# D. Risco de Dimensionalidade (Features por Observação)
df_final['Dimensionality_Ratio'] = np.where(
    df_final['Rows_Count'] > 0,
    round(df_final['Columns_Count'] / df_final['Rows_Count'], 6),
    0
)

# E. Proporção Categórica (% de colunas que não são números)
df_final['Categorical_Ratio_%'] = np.where(
    df_final['Columns_Count'] > 0,
    round((df_final['Categorical_Cols'] / df_final['Columns_Count']) * 100, 2),
    0
)

# F. Indicador de Justiça Algorítmica (Contém dados demográficos?)
# Se for uma lista e tiver tamanho maior que 0, é True. Caso contrário, False.
df_final['Has_Sensitive_Data'] = df_final['Sensitive_Columns'].apply(
    lambda x: True if isinstance(x, (list, np.ndarray)) and len(x) > 0 else False
)

# G. Contagem exata de atributos sensíveis encontrados
# Conta quantos itens existem na lista. Se não for lista ou estiver vazia, retorna 0.
df_final['Sensitive_Count'] = df_final['Sensitive_Columns'].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

# =====================================================================
# 4. ORDENAÇÃO E EXPORTAÇÃO
# =====================================================================

# H. Ordenar pelo ID original para manter a integridade com o seu mapeamento
df_final = df_final.sort_values(by='id').reset_index(drop=True)

# I. Exportar para Excel (Requer openpyxl)
caminho_saida = 'Data/06_Datasets_Metadata_Final.xlsx'
df_final.to_excel(caminho_saida, index=False, engine='openpyxl')

print(f"\n| * * * Sucesso Absoluto! * * * |")
print(f"| Tabela final consolidada, limpa e enriquecida com {len(df_final)} datasets.")
print(f"| Arquivo salvo em: {caminho_saida}")


Consolidando as bases de dados...
Consolidando as bases de dados extraídas...

| * * * Sucesso Absoluto! * * * |
| Tabela final consolidada, limpa e enriquecida com 49 datasets.
| Arquivo salvo em: Data/06_Datasets_Metadata_Final.xlsx
